In [137]:
# !pip install koreanize_matplotlib

import warnings
import koreanize_matplotlib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as  plt
from matplotlib_venn import venn2
import plotly.express as px
import plotly.graph_objects as go
import ast


# 그래프 해상도 높이기
try:
    %config InlineBackend.figure_format = 'retina'
except Exception as e:
    print(f'💩 {e}')



# 경고 무시
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)  # 출력할 너비를 넉넉하게 조정
pd.set_option('display.expand_frame_repr', False)  # 옆으로 길어져도 줄바꿈 없이 출력
pd.set_option('display.max_colwidth', None)  # 긴 문자열도 생략 없이 출력

try:
    from google.colab import drive
    drive.mount('/content/drive')

    import os
    os.chdir('/content/drive/MyDrive/파트4')
    print('✅ Succesful access google_drive_directory')
    
except Exception as e:
    print('🤗 Hello vscode')
        
## get_df 함수
def get_df(db_name, table_name):
    if table_name in ['accounts_user', 'accounts_blockrecord', 'hackle_events']:
        table_name = pd.read_parquet(
            f"gs://high_project/{db_name}/{table_name}.parquet", 
            storage_options={'token' : API_KEY_PATH})
    else:
        table_name = pd.read_csv(
            f"gs://high_project/{db_name}/{table_name}.csv",
            storage_options={'token' : API_KEY_PATH}
            )
    return table_name
    

## literal_eval 형변환 함수
def to_literal_eval(df, column):
    return  df[column].apply(lambda x: ast.literal_eval(x) if x != '[]' else [])


# 리스트 내에 드랍 유저가 있는지 확인하는 함수
def find_drop_users(df, column):
    drop_users = [831956, 1580627, 1580689, 1580626, 995177]
    print(f'{column}:')
    for i in drop_users:
        count_drop_rows = len(df[df[column].apply(lambda x: i in x)])
        if count_drop_rows != 0:
            print(f"‼️ 관리자 {i}가 포함된 행 {count_drop_rows}개 존재")
        else:
            print(f"✅ 관리자 {i} 포함행 없음")

def get_drop_users(df, user_column):
    try:
        drop_users = [831956, 1580627, 1580689, 1580626, 995177]     
        print(f'✅ {user_column} 컬럼 관리자 유저 삭제 완')
        return df[~df[user_column].isin(drop_users)]

    except Exception as e:
        print(f'💩 {e}')    
            
# 데이트타임형으로 변환 및 기간 전처리
def set_datetime(df, column):
    df[column] =  pd.to_datetime(df[column])
    print(f'✅ {column}데이트 타입 형변환 및 기간 전처리 완료')
    
    
# 데이트타임형으로 변환 및 기간 전처리
def get_datetime(df, column):
    try:
        df[column] =  pd.to_datetime(df[column])
        print(f'✅ {column} 데이트 타입 형변환 완')
    except Exception as e:
        print(f'💩 {e}')

    return df[df[column] < '2023-09-01']


            
API_KEY_PATH ='/home/project_yujin/API_KEY/sprintda03-yujin.json'

🤗 Hello vscode


## question_id별 질문내용

In [94]:
polls_question = get_df('votes', 'polls_question')
polls_question.head()

,id,question_text,created_at
0,99,가장 신비한 매력이 있는 사람은?,2023-03-31 15:22:53
1,100,"""이 사람으로 한 번 살아보고 싶다"" 하는 사람은?",2023-03-31 15:22:53
2,101,미래의 틱톡커는?,2023-03-31 15:22:54
3,102,여기서 제일 특이한 친구는?,2023-03-31 15:22:54
4,103,가장 지켜주고 싶은 사람은?,2023-03-31 15:22:55


## 주별로 많이보낸질문 top10 (polls_questionpiece)

In [102]:
# 데이터 불러오기 및 컬럼순서 별경
polls_questionpiece = get_df('votes', 'polls_questionpiece')
polls_questionpiece = polls_questionpiece[['id' ,'question_id', 'is_voted', 'is_skipped', 'created_at']]

# 시간타입 변경 및 8월까지 필터링
polls_questionpiece = get_datetime(polls_questionpiece, 'created_at')

# 년, 월, 주차 컬럼 추가
polls_questionpiece['created_year'] = polls_questionpiece['created_at'].dt.isocalendar().year
polls_questionpiece['created_month'] = polls_questionpiece['created_at'].dt.to_period('M').astype('str')
# 월별 몇 번째 주인지 계산
polls_questionpiece['week_of_month'] = polls_questionpiece['created_at'].apply(
    lambda x: ((x.day - 1) // 7) + 1
)

print(len(polls_questionpiece))
polls_questionpiece.head()

✅ created_at 데이트 타입 형변환 완
1261394


,id,question_id,is_voted,is_skipped,created_at,created_year,created_month,week_of_month
0,998458,252,1,0,2023-04-28 12:27:22,2023,2023-04,4
1,998459,244,1,0,2023-04-28 12:27:22,2023,2023-04,4
2,998460,183,1,0,2023-04-28 12:27:22,2023,2023-04,4
3,998461,101,1,0,2023-04-28 12:27:22,2023,2023-04,4
4,998462,209,1,0,2023-04-28 12:27:22,2023,2023-04,4


In [107]:
polls_questionpiece.query('is_voted == 1')

,id,question_id,is_voted,is_skipped,created_at,created_year,created_month,week_of_month
0,998458,252,1,0,2023-04-28 12:27:22,2023,2023-04,4
1,998459,244,1,0,2023-04-28 12:27:22,2023,2023-04,4
2,998460,183,1,0,2023-04-28 12:27:22,2023,2023-04,4
3,998461,101,1,0,2023-04-28 12:27:22,2023,2023-04,4
4,998462,209,1,0,2023-04-28 12:27:22,2023,2023-04,4
...,...,...,...,...,...,...,...,...
1261389,207182135,845,1,0,2023-08-31 16:07:58,2023,2023-08,5
1261390,207182137,3935,1,0,2023-08-31 16:07:58,2023,2023-08,5
1261391,207182138,4901,1,0,2023-08-31 16:07:58,2023,2023-08,5
1261392,207182139,1961,1,0,2023-08-31 16:07:58,2023,2023-08,5


In [113]:
top10_by_week_voted = (
    polls_questionpiece.query('is_voted == 1')
    .groupby(['created_month', 'week_of_month', 'question_id'])
    .size()
    .reset_index(name='count')
    .sort_values(['created_month', 'week_of_month', 'count'], ascending=[True, True, False])
    
    .groupby(['created_month', 'week_of_month'])
    .head(10)  # 각 주별 상위 10개 질문
    
    .assign(rank=lambda df: df.groupby(['created_month', 'week_of_month']).cumcount() + 1) # 각행에서 몇번째 행인지 +1
    .reset_index(drop=True)
)

top10_by_week_voted = top10_by_week_voted[['created_month', 'week_of_month', 'rank', 'question_id', 'count']]
top10_by_week_voted

,created_month,week_of_month,rank,question_id,count
0,2023-04,4,1,272,33
1,2023-04,4,2,118,30
2,2023-04,4,3,139,30
3,2023-04,4,4,212,28
4,2023-04,4,5,101,26
...,...,...,...,...,...
215,2023-08,5,6,456,1
216,2023-08,5,7,468,1
217,2023-08,5,8,493,1
218,2023-08,5,9,612,1


In [103]:
top10_by_month_week = (
    polls_questionpiece.groupby(['created_month', 'week_of_month', 'question_id'])
      .size()
      .reset_index(name='count')
      .sort_values(['created_month', 'week_of_month', 'count'], ascending=[True, True, False])
      
      .groupby(['created_month', 'week_of_month'])
      .head(10)
      
      .assign(rank=lambda df: df.groupby(['created_month', 'week_of_month']).cumcount() + 1) # 각행에서 몇번째 행인지 +1
      .reset_index(drop=True)
)

top10_by_month_week = top10_by_month_week[['created_month', 'week_of_month', 'rank', 'question_id', 'count']]
top10_by_month_week

,created_month,week_of_month,rank,question_id,count
0,2023-04,4,1,272,33
1,2023-04,4,2,118,30
2,2023-04,4,3,139,30
3,2023-04,4,4,212,28
4,2023-04,4,5,101,26
...,...,...,...,...,...
215,2023-08,5,6,221,1
216,2023-08,5,7,263,1
217,2023-08,5,8,319,1
218,2023-08,5,9,376,1


In [120]:
# top10_by_week_voted[(top10_by_week_voted['count'] != top10_by_month_week['count'])]
# top10_by_month_week[(top10_by_week_voted['count'] != top10_by_month_week['count'])]

In [ ]:
top10_by_month_week = pd.merge(top10_by_month_week, polls_question,
        left_on='question_id', right_on='id')
polls_questionpiece.groupby('created_month')['question_id'].value_counts().unstack(fill_value=0)

## 초성확인이 많이 된 질문 top10

In [ ]:
# 데이터 불러오기 및 컬럼순서 별경
accounts_userquestionrecord = get_df('votes', 'accounts_userquestionrecord')
accounts_userquestionrecord = accounts_userquestionrecord[['id', 'user_id', 'chosen_user_id', 'question_id', 'question_piece_id', \
    'status' ,'answer_status', 'answer_updated_at', 'has_read', 'opened_times', 'report_count', 'created_at']]

# 관리저 유저 드랍
accounts_userquestionrecord = get_drop_users(accounts_userquestionrecord, 'user_id')
accounts_userquestionrecord = get_drop_users(accounts_userquestionrecord, 'chosen_user_id')


# 시간타입 변경 및 8월까지 필터링
accounts_userquestionrecord = get_datetime(accounts_userquestionrecord, 'answer_updated_at')
accounts_userquestionrecord = get_datetime(accounts_userquestionrecord, 'created_at')


# 상태컬럼, 응답상태컬럼 한글로 매핑
accounts_userquestionrecord['status'] = accounts_userquestionrecord['status'].replace({'C':'닫힘'}).replace({'I':'초성열림'}).replace({'B':'차단'})
accounts_userquestionrecord['answer_status'] = accounts_userquestionrecord['answer_status'].replace({'N':'미답변'}).replace({'P':'비공개'}).replace({'A':'공개'})

# 년, 월, 주차 컬럼 추가
accounts_userquestionrecord['updated_year'] = accounts_userquestionrecord['answer_updated_at'].dt.isocalendar().year
accounts_userquestionrecord['updated_month'] = accounts_userquestionrecord['answer_updated_at'].dt.to_period('M').astype('str')
# 월별 몇 번째 주인지 계산
accounts_userquestionrecord['week_of_month'] = accounts_userquestionrecord['answer_updated_at'].apply(
    lambda x: ((x.day - 1) // 7) + 1
)

accounts_userquestionrecord.head()

✅ user_id 컬럼 관리자 유저 삭제 완
✅ chosen_user_id 컬럼 관리자 유저 삭제 완
✅ answer_updated_at 데이트 타입 형변환 완
✅ created_at 데이트 타입 형변환 완


,id,user_id,chosen_user_id,question_id,question_piece_id,status,answer_status,answer_updated_at,has_read,opened_times,report_count,created_at
0,771777,849436,849469,252,998458,닫힘,미답변,2023-04-28 12:27:49,0,0,0,2023-04-28 12:27:49
1,771800,849436,849446,244,998459,닫힘,미답변,2023-04-28 12:28:02,0,0,0,2023-04-28 12:28:02
2,771812,849436,849454,183,998460,닫힘,미답변,2023-04-28 12:28:09,1,0,0,2023-04-28 12:28:09
3,771828,849436,847375,101,998461,닫힘,미답변,2023-04-28 12:28:16,0,0,0,2023-04-28 12:28:16
4,771851,849436,849477,209,998462,닫힘,미답변,2023-04-28 12:28:26,1,0,0,2023-04-28 12:28:26


In [141]:
accounts_userquestionrecord['status'].unique()

array(['닫힘', '초성열림', '차단'], dtype=object)

In [ ]:
finded_initial = accounts_userquestionrecord.query('status == "초성열림" | status == "I"')

In [ ]:
finded_initial.groupby()

,id,user_id,chosen_user_id,question_id,question_piece_id,status,answer_status,answer_updated_at,has_read,opened_times,report_count,created_at,updated_year,updated_month,week_of_month
10,771940,849436,849634,297,998465,초성열림,미답변,2023-04-28 12:29:13,1,1,0,2023-04-28 12:29:13,2023,2023-04,4
34,772123,849452,849489,209,999059,초성열림,미답변,2023-04-28 12:30:48,0,1,0,2023-04-28 12:30:48,2023,2023-04,4
35,772126,849438,849488,257,998594,초성열림,미답변,2023-04-28 12:30:49,1,2,0,2023-04-28 12:30:49,2023,2023-04,4
95,772488,849445,849446,245,999492,초성열림,미답변,2023-04-28 12:33:33,0,1,0,2023-04-28 12:33:33,2023,2023-04,4
117,772628,849477,849436,253,998774,초성열림,미답변,2023-04-28 12:34:17,0,1,0,2023-04-28 12:34:17,2023,2023-04,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1214198,160861052,1577437,1577440,612,207171000,초성열림,미답변,2023-08-30 15:07:19,1,1,0,2023-08-30 15:07:19,2023,2023-08,5
1214200,160861057,1577437,886028,2175,207171004,초성열림,미답변,2023-08-30 15:07:39,1,3,0,2023-08-30 15:07:39,2023,2023-08,5
1214204,160861061,1577437,1113407,3001,207171008,초성열림,미답변,2023-08-30 15:08:13,1,1,0,2023-08-30 15:08:13,2023,2023-08,5
1214215,160862591,1577437,1577440,376,207171879,초성열림,미답변,2023-08-31 02:58:13,1,1,0,2023-08-31 02:58:13,2023,2023-08,5


In [149]:
top10_by_week_initial_check = (
    accounts_userquestionrecord.query('status == "초성열림" | status == "I"') 
    .groupby(['updated_month', 'week_of_month', 'question_id'])
    .size()
    .reset_index(name='count')
    .sort_values(['updated_month', 'week_of_month', 'count'], ascending=[True, True, False])
    
    .groupby(['updated_month', 'week_of_month'])
    .head(10)
    
    .assign(initial_check_rank=lambda df: df.groupby(['updated_month', 'week_of_month']).cumcount() + 1) # 각행에서 몇번째 행인지 +1
    .reset_index(drop=True)
)

top10_by_week_initial_check = top10_by_week_initial_check[['updated_month', 'week_of_month', 'initial_check_rank', 'question_id', 'count']]
top10_by_week_initial_check

,updated_month,week_of_month,initial_check_rank,question_id,count
0,2023-04,4,1,136,3
1,2023-04,4,2,203,3
2,2023-04,4,3,105,2
3,2023-04,4,4,134,2
4,2023-04,4,5,135,2
...,...,...,...,...,...
213,2023-08,5,4,1425,1
214,2023-08,5,5,2175,1
215,2023-08,5,6,3001,1
216,2023-08,5,7,3146,1


In [150]:
top10_by_week_initial_check['initial_check_rank'].value_counts()

initial_check_rank
1     22
2     22
3     22
4     22
5     22
6     22
7     22
8     22
9     21
10    21
Name: count, dtype: int64

In [154]:
top10_by_week_initial_check[top10_by_week_initial_check[['updated_month', 'count']].duplicated(keep=False)].head(30)

,updated_month,week_of_month,initial_check_rank,question_id,count
0,2023-04,4,1,136,3
1,2023-04,4,2,203,3
2,2023-04,4,3,105,2
3,2023-04,4,4,134,2
4,2023-04,4,5,135,2
5,2023-04,4,6,138,2
6,2023-04,4,7,180,2
7,2023-04,4,8,183,2
8,2023-04,4,9,193,2
9,2023-04,4,10,195,2


In [ ]:
top10_by_week_initial_check = (
    polls_questionpiece.query('is_initial_checked == 1')  # 초성 확인된 데이터만 필터링
    .groupby(['created_week_num', 'question_id'])  # 주별, 질문별로 그룹화
    .size()  # 각 그룹의 크기(초성 확인 수)를 계산
    .reset_index(name='initial_check_count')  # 초기 확인 수 저장
    .sort_values(['created_week_num', 'initial_check_count'], ascending=[True, False])  # 주별로 초성 확인 수 내림차순 정렬
    .groupby('created_week_num')  # 주별로 다시 그룹화
    .head(10)  # 각 주별 상위 10개 질문
    .assign(rank=lambda df: df.groupby('created_week_num').cumcount() + 1)  # 순위 추가
    .reset_index(drop=True)  # 인덱스를 재설정
)